In [1]:
!pip install transformers datasets accelerate scikit-learn sentencepiece protobuf -q

In [2]:
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/deberta-v3-base.zip"
cache_dir = os.path.expanduser("~/.cache/huggingface/hub/")
os.makedirs(cache_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    for info in z.infolist():
        fixed_name = info.filename.replace("\\", "/")
        out_path = os.path.join(cache_dir, fixed_name)
        if info.is_dir() or fixed_name.endswith("/"):
            os.makedirs(out_path, exist_ok=True)
        elif info.file_size == 0:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
        else:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                dst.write(src.read())

model_cache = os.path.join(cache_dir, "models--microsoft--deberta-v3-base")
print(f"Model extracted to: {model_cache}")
print(f"Contents: {os.listdir(model_cache)}")

Mounted at /content/drive
Model extracted to: /root/.cache/huggingface/hub/models--microsoft--deberta-v3-base
Contents: ['refs', 'snapshots', '.no_exist']


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import os
import gc
import re
import nltk
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_sim

os.environ["HF_HUB_OFFLINE"] = "1"

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [4]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

df_raw = pd.read_csv('ExioNAICS.csv')

naics_corpus = df_raw[['NAICS Code', 'NAICS Title', 'Description']].drop_duplicates(subset='NAICS Code').copy()
naics_corpus['NAICS Code'] = naics_corpus['NAICS Code'].astype(str)
naics_corpus['clean_text'] = (naics_corpus['NAICS Title'] + ' ' + naics_corpus['Description'].fillna('')).apply(preprocess_text)
naics_corpus = naics_corpus.reset_index(drop=True)

corpus_texts = naics_corpus['clean_text'].tolist()
code_to_idx = {code: i for i, code in enumerate(naics_corpus['NAICS Code'])}

df = pd.read_csv('ExioNAICS_preprocessed.csv')
df['NAICS Code'] = df['NAICS Code'].astype(str)
df['naics_idx'] = df['NAICS Code'].map(code_to_idx)

missing = df['naics_idx'].isna().sum()
if missing > 0:
    df = df.dropna(subset=['naics_idx']).reset_index(drop=True)
df['naics_idx'] = df['naics_idx'].astype(int)

print(f"Corpus: {len(corpus_texts)} NAICS codes")
print(f"Dataset: {len(df)} samples")

print("\nBuilding TF-IDF index for candidate retrieval...")
tfidf = TfidfVectorizer(max_features=10000, sublinear_tf=True)
all_clean = corpus_texts + df['clean_description'].tolist()
tfidf_matrix = tfidf.fit_transform(all_clean)
corpus_tfidf = tfidf_matrix[:len(corpus_texts)]
query_tfidf = tfidf_matrix[len(corpus_texts):]

print("Computing similarity matrix...")
sim_matrix = sklearn_cosine_sim(query_tfidf, corpus_tfidf)
tfidf_top100 = np.argsort(-sim_matrix, axis=1)[:, :100]

tfidf_r50 = np.mean([df['naics_idx'].iloc[i] in tfidf_top100[i, :50] for i in range(len(df))])
tfidf_r100 = np.mean([df['naics_idx'].iloc[i] in tfidf_top100[i] for i in range(len(df))])
print(f"\nTF-IDF Recall@50:  {tfidf_r50:.4f}")
print(f"TF-IDF Recall@100: {tfidf_r100:.4f}")

del sim_matrix
gc.collect()

Corpus: 1115 NAICS codes
Dataset: 20535 samples

Building TF-IDF index for candidate retrieval...
Computing similarity matrix...

TF-IDF Recall@50:  0.5639
TF-IDF Recall@100: 0.6360


60

In [5]:
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 192
BATCH_SIZE = 4
NUM_HARD_NEG = 4
NUM_RANDOM_NEG = 3
NUM_NEGATIVES = NUM_HARD_NEG + NUM_RANDOM_NEG
NUM_EPOCHS = 20
LEARNING_RATE = 1.5e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42
VAL_RATIO = 0.1
PATIENCE = 7
EVAL_TOP_K = 50
FULL_RERANK_SUBSET = 500

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"=== DeBERTa-v3-BASE Cross-Encoder (Mixed Negatives) ===")
print(f"  Model:        {MODEL_NAME}")
print(f"  Max length:   {MAX_LENGTH}")
print(f"  Batch size:   {BATCH_SIZE} queries x {NUM_NEGATIVES+1} candidates = {BATCH_SIZE*(NUM_NEGATIVES+1)} seq/step")
print(f"  Negatives:    {NUM_HARD_NEG} hard (TF-IDF) + {NUM_RANDOM_NEG} random = {NUM_NEGATIVES} total")
print(f"  Epochs:       {NUM_EPOCHS}")
print(f"  LR:           {LEARNING_RATE}")
print(f"  Patience:     {PATIENCE}")
print(f"  Epoch eval:   full rerank on {FULL_RERANK_SUBSET} val samples + TF-IDF@{EVAL_TOP_K} on all")

=== DeBERTa-v3-BASE Cross-Encoder (Mixed Negatives) ===
  Model:        microsoft/deberta-v3-base
  Max length:   192
  Batch size:   4 queries x 8 candidates = 32 seq/step
  Negatives:    4 hard (TF-IDF) + 3 random = 7 total
  Epochs:       20
  LR:           1.5e-05
  Patience:     7
  Epoch eval:   full rerank on 500 val samples + TF-IDF@50 on all


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, torch_dtype=torch.float32
).to(device)

print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

test_enc = tokenizer("company description", "naics industry description", return_tensors="pt", max_length=MAX_LENGTH, truncation=True, padding=True)
test_enc = {k: v.to(device) for k, v in test_enc.items() if k in ['input_ids', 'attention_mask']}
with torch.no_grad():
    test_out = model(**test_enc)
print(f"Output shape: {test_out.logits.shape}")
print(f"NaN: {torch.isnan(test_out.logits).any().item()}")

The tokenizer you are loading from 'microsoft/deberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight        

Total params: 184,422,913
Output shape: torch.Size([1, 1])
NaN: False


In [7]:
class CrossEncoderDataset(Dataset):
    def __init__(self, descriptions, naics_indices, corpus_texts, tfidf_candidates,
                 tokenizer, max_length, num_hard_neg=4, num_random_neg=3):
        self.descriptions = descriptions
        self.naics_indices = naics_indices
        self.corpus_texts = corpus_texts
        self.tfidf_candidates = tfidf_candidates
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.num_hard_neg = num_hard_neg
        self.num_random_neg = num_random_neg
        self.num_classes = len(corpus_texts)

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        query = self.descriptions[idx]
        pos_idx = self.naics_indices[idx]

        hard_pool = [int(c) for c in self.tfidf_candidates[idx] if int(c) != pos_idx]
        if len(hard_pool) >= self.num_hard_neg:
            hard_negs = random.sample(hard_pool[:50], self.num_hard_neg)
        else:
            hard_negs = list(hard_pool)

        used = {pos_idx} | set(hard_negs)
        random_negs = []
        while len(random_negs) < self.num_random_neg:
            r = random.randint(0, self.num_classes - 1)
            if r not in used:
                random_negs.append(r)
                used.add(r)

        while len(hard_negs) < self.num_hard_neg:
            r = random.randint(0, self.num_classes - 1)
            if r not in used:
                hard_negs.append(r)
                used.add(r)

        candidate_indices = [pos_idx] + hard_negs + random_negs

        input_ids_list = []
        attention_mask_list = []

        for c_idx in candidate_indices:
            enc = self.tokenizer(
                query, self.corpus_texts[c_idx],
                max_length=self.max_length, truncation=True,
                padding='max_length', return_tensors='pt'
            )
            input_ids_list.append(enc['input_ids'].squeeze(0))
            attention_mask_list.append(enc['attention_mask'].squeeze(0))

        return {
            'input_ids': torch.stack(input_ids_list),
            'attention_mask': torch.stack(attention_mask_list),
            'naics_idx': pos_idx,
        }

print("CrossEncoderDataset defined (mixed hard + random negatives)")

CrossEncoderDataset defined (mixed hard + random negatives)


In [8]:
label_counts = df['naics_idx'].value_counts()
valid_labels = label_counts[label_counts >= 2].index
mask = df['naics_idx'].isin(valid_labels)
df_filtered = df[mask].reset_index(drop=True)
tfidf_filtered = tfidf_top100[mask.values]

train_idx, val_idx = train_test_split(
    np.arange(len(df_filtered)), test_size=VAL_RATIO, random_state=SEED, stratify=df_filtered['naics_idx']
)

train_dataset = CrossEncoderDataset(
    df_filtered['clean_description'].iloc[train_idx].tolist(),
    df_filtered['naics_idx'].iloc[train_idx].tolist(),
    corpus_texts,
    tfidf_filtered[train_idx],
    tokenizer, MAX_LENGTH, NUM_HARD_NEG, NUM_RANDOM_NEG
)

val_descriptions = df_filtered['clean_description'].iloc[val_idx].tolist()
val_naics_indices = df_filtered['naics_idx'].iloc[val_idx].tolist()
val_tfidf = tfidf_filtered[val_idx]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print(f"After filtering: {len(df_filtered)} samples, {df_filtered['naics_idx'].nunique()} codes")
print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_descriptions)} samples")

val_tfidf_r50 = np.mean([val_naics_indices[i] in val_tfidf[i, :EVAL_TOP_K] for i in range(len(val_naics_indices))])
print(f"Val TF-IDF Recall@{EVAL_TOP_K}: {val_tfidf_r50:.4f} (ceiling for reranking eval)")

After filtering: 20533 samples, 1113 codes
Train: 18479 samples, 4620 batches
Val:   2054 samples
Val TF-IDF Recall@50: 0.5594 (ceiling for reranking eval)


In [9]:
@torch.no_grad()
def evaluate_rerank(model, tokenizer, val_descriptions, val_labels, corpus_texts,
                    val_tfidf, max_length, eval_k=50, score_batch=64):
    model.eval()
    n = len(val_descriptions)
    top1, top5, top10 = 0, 0, 0
    recall_count = 0

    for i in range(n):
        query = val_descriptions[i]
        true_label = val_labels[i]
        candidates = val_tfidf[i, :eval_k].tolist()

        if true_label in candidates:
            recall_count += 1

        scores = []
        for j in range(0, len(candidates), score_batch):
            batch_cands = candidates[j:j+score_batch]
            batch_docs = [corpus_texts[int(c)] for c in batch_cands]
            enc = tokenizer(
                [query] * len(batch_docs), batch_docs,
                max_length=max_length, truncation=True, padding=True,
                return_tensors='pt'
            )
            enc = {k: v.to(device) for k, v in enc.items() if k in ['input_ids', 'attention_mask']}
            logits = model(**enc).logits.squeeze(-1)
            scores.extend(logits.cpu().tolist())

        ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
        ranked_indices = [int(r[0]) for r in ranked]

        if ranked_indices[0] == true_label: top1 += 1
        if true_label in ranked_indices[:5]: top5 += 1
        if true_label in ranked_indices[:10]: top10 += 1

    return {
        'tfidf_recall': recall_count / n,
        'top1': top1 / n,
        'top5': top5 / n,
        'top10': top10 / n,
    }


@torch.no_grad()
def evaluate_full_rerank(model, tokenizer, val_descriptions, val_labels, corpus_texts,
                         max_length, score_batch=128):
    model.eval()
    n = len(val_descriptions)
    num_classes = len(corpus_texts)
    top1, top5, top10 = 0, 0, 0

    for i in range(n):
        query = val_descriptions[i]
        true_label = val_labels[i]
        all_scores = []

        for j in range(0, num_classes, score_batch):
            batch_docs = corpus_texts[j:j+score_batch]
            enc = tokenizer(
                [query] * len(batch_docs), batch_docs,
                max_length=max_length, truncation=True, padding=True,
                return_tensors='pt'
            )
            enc = {k: v.to(device) for k, v in enc.items() if k in ['input_ids', 'attention_mask']}
            logits = model(**enc).logits.squeeze(-1)
            all_scores.append(logits.cpu())

        scores = torch.cat(all_scores, dim=0)
        topk = scores.topk(10).indices.tolist()

        if topk[0] == true_label: top1 += 1
        if true_label in topk[:5]: top5 += 1
        if true_label in topk[:10]: top10 += 1

        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{n} | Top-1: {top1/(i+1):.4f}")

    return {'top1': top1 / n, 'top5': top5 / n, 'top10': top10 / n}

print("Evaluation functions defined")
print(f"  Per-epoch: rerank TF-IDF top-{EVAL_TOP_K} candidates")
print(f"  Final: rerank all {len(corpus_texts)} candidates")

Evaluation functions defined
  Per-epoch: rerank TF-IDF top-50 candidates
  Final: rerank all 1115 candidates


In [ ]:
from transformers import get_cosine_schedule_with_warmup
import shutil

CHECKPOINT_DIR = "/content/drive/MyDrive/cross_encoder_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_top1 = 0.0
best_epoch = 0
patience_counter = 0
epoch_log = []
start_epoch = 1

checkpoint_path = os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pt")
if os.path.exists(checkpoint_path):
    print("Found checkpoint on Google Drive, resuming...")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_top1 = ckpt['best_top1']
    best_epoch = ckpt['best_epoch']
    patience_counter = ckpt['patience_counter']
    epoch_log = ckpt['epoch_log']
    print(f"  Resumed from epoch {ckpt['epoch']}, best Full Top-1: {best_top1:.4f} (epoch {best_epoch})")
    print(f"  Patience: {patience_counter}/{PATIENCE}")
    del ckpt
    torch.cuda.empty_cache()
else:
    print("No checkpoint found, starting from scratch.")

subset_idx = random.sample(range(len(val_descriptions)), min(FULL_RERANK_SUBSET, len(val_descriptions)))
subset_descs = [val_descriptions[i] for i in subset_idx]
subset_labels = [val_naics_indices[i] for i in subset_idx]

print(f"\nTotal steps: {total_steps}")
print(f"Warmup: {warmup_steps} steps")
print(f"Training epochs {start_epoch} to {NUM_EPOCHS}")
print(f"Full rerank eval subset: {len(subset_descs)} samples")
print(f"\n{'='*95}")
print(f"{'Ep':>3} {'TrLoss':>8} {'Full1':>7} {'Full5':>7} {'Full10':>7} {'TfR50':>6} {'Tf1':>6} {'Tf5':>6} {'LR':>10}")
print(f"{'='*95}")

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch in train_loader:
        bsz = batch['input_ids'].size(0)
        num_cands = batch['input_ids'].size(1)

        input_ids = batch['input_ids'].view(-1, MAX_LENGTH).to(device)
        attention_mask = batch['attention_mask'].view(-1, MAX_LENGTH).to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(-1)
        scores = logits.view(bsz, num_cands)
        labels = torch.zeros(bsz, dtype=torch.long, device=device)
        loss = F.cross_entropy(scores, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches
    lr = scheduler.get_last_lr()[0]

    full_m = evaluate_full_rerank(
        model, tokenizer, subset_descs, subset_labels,
        corpus_texts, MAX_LENGTH, score_batch=128
    )

    tfidf_m = evaluate_rerank(
        model, tokenizer, val_descriptions, val_naics_indices,
        corpus_texts, val_tfidf, MAX_LENGTH, eval_k=EVAL_TOP_K
    )

    epoch_log.append({
        'epoch': epoch, 'train_loss': avg_loss, 'lr': lr,
        'full_top1': full_m['top1'], 'full_top5': full_m['top5'], 'full_top10': full_m['top10'],
        'tfidf_recall': tfidf_m['tfidf_recall'],
        'tfidf_top1': tfidf_m['top1'], 'tfidf_top5': tfidf_m['top5'], 'tfidf_top10': tfidf_m['top10'],
    })

    print(f"{epoch:>3} {avg_loss:>8.4f} {full_m['top1']:>7.4f} {full_m['top5']:>7.4f} {full_m['top10']:>7.4f} "
          f"{tfidf_m['tfidf_recall']:>6.3f} {tfidf_m['top1']:>6.3f} {tfidf_m['top5']:>6.3f} {lr:>10.2e}")

    track_metric = full_m['top1']
    if track_metric > best_top1:
        best_top1 = track_metric
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), "results/best_model.pt")
        shutil.copy("results/best_model.pt", os.path.join(CHECKPOINT_DIR, "best_model.pt"))
    else:
        patience_counter += 1

    pd.DataFrame(epoch_log).to_csv("results/cross_encoder_epoch_log.csv", index=False)

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_top1': best_top1,
        'best_epoch': best_epoch,
        'patience_counter': patience_counter,
        'epoch_log': epoch_log,
    }, checkpoint_path)
    print(f"      [Checkpoint saved to Drive]")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\n{'='*95}")
print(f"Best Full-Rerank Top-1 (subset): {best_top1:.4f} at epoch {best_epoch}")

best_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, weights_only=True))
else:
    model.load_state_dict(torch.load("results/best_model.pt", weights_only=True))
print("Loaded best model.")

Total steps: 92400
Warmup: 9240 steps
Full rerank eval subset: 500 samples

 Ep   TrLoss   Full1   Full5  Full10  TfR50    Tf1    Tf5         LR
  200/500 | Top-1: 0.0150
  400/500 | Top-1: 0.0200
  1   2.0675  0.0180  0.0880  0.1420  0.559  0.040  0.187   7.50e-06
  200/500 | Top-1: 0.1250
  400/500 | Top-1: 0.1425
  2   1.5071  0.1380  0.3720  0.5080  0.559  0.153  0.375   1.50e-05
  200/500 | Top-1: 0.1300
  400/500 | Top-1: 0.1775
  3   1.2262  0.1760  0.4380  0.5540  0.559  0.168  0.397   1.49e-05


In [ ]:
print("=== Final Evaluation: Reranking ALL 1,115 NAICS codes ===")
print("(This takes ~15 minutes)\n")

full_metrics = evaluate_full_rerank(
    model, tokenizer, val_descriptions, val_naics_indices,
    corpus_texts, MAX_LENGTH, score_batch=128
)

tfidf_metrics = evaluate_rerank(
    model, tokenizer, val_descriptions, val_naics_indices,
    corpus_texts, val_tfidf, MAX_LENGTH, eval_k=EVAL_TOP_K
)

print(f"\n=== Results ===")
print(f"\n  Full Rerank (all {len(corpus_texts)} candidates):")
print(f"    Top-1:  {full_metrics['top1']:.4f}")
print(f"    Top-5:  {full_metrics['top5']:.4f}")
print(f"    Top-10: {full_metrics['top10']:.4f}")
print(f"\n  TF-IDF@{EVAL_TOP_K} Rerank:")
print(f"    Recall: {tfidf_metrics['tfidf_recall']:.4f}")
print(f"    Top-1:  {tfidf_metrics['top1']:.4f}")
print(f"    Top-5:  {tfidf_metrics['top5']:.4f}")
print(f"    Top-10: {tfidf_metrics['top10']:.4f}")
print(f"\n  Best epoch: {best_epoch}")

results = {
    "method": "DeBERTa-v3-base Cross-Encoder + Mixed Negatives + TF-IDF Retrieval",
    "config": {
        "model": MODEL_NAME, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,
        "num_negatives": NUM_NEGATIVES, "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "num_epochs_trained": best_epoch,
        "eval_top_k": EVAL_TOP_K,
    },
    "full_rerank_results": full_metrics,
    "tfidf_rerank_results": tfidf_metrics,
}

with open("results/cross_encoder_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

import zipfile, glob
with zipfile.ZipFile("results.zip", "w") as zf:
    for f in glob.glob("results/*.json") + glob.glob("results/*.csv") + glob.glob("results/*.pt"):
        zf.write(f)
        print(f"  Added: {f}")

from google.colab import files
files.download("results.zip")
print("\nDownloaded results.zip")